In [58]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_class_weight

In [59]:
df = pd.read_excel(r"data_phase3_2.xlsx")

df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')
df = df.drop(columns=['year', 'month', 'day', 'dayofweek', 'isweekend'])
df.columns

Index(['order_id', 'customer_id', 'order_priority', 'market', 'sales',
       'quantity', 'discount', 'profit', 'shipping_cost', 'ship_mode',
       'region', 'country', 'city', 'isreturned'],
      dtype='object')

In [60]:
df

,order_id,customer_id,order_priority,market,sales,quantity,discount,profit,shipping_cost,ship_mode,region,country,city,isreturned
0,CA-2014-146780,CV-12805,Medium,US,41.959999,2,0.0,10.909600,4.180000,Standard Class,east,united states,new york city,No
1,CA-2013-138520,JL-15505,Medium,US,8.260000,2,0.0,3.799600,0.690000,Standard Class,east,united states,new york city,No
2,CA-2014-144904,KW-16435,Medium,US,20.700001,2,0.0,9.936000,0.740000,Standard Class,east,united states,new york city,No
3,CA-2014-144904,KW-16435,Medium,US,5.560000,2,0.0,1.445600,0.420000,Standard Class,east,united states,new york city,No
4,CA-2013-123274,GT-14710,Medium,US,44.459999,2,0.0,14.671800,3.970000,Standard Class,east,united states,new york city,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
49665,CA-2014-128734,JL-15175,Medium,US,842.375977,3,0.2,105.296997,70.220001,Standard Class,west,united states,chandler,No
49666,US-2012-163279,JD-16150,Medium,US,1487.979980,3,0.2,185.996994,163.539993,Standard Class,west,united states,san diego,No
49667,CA-2013-148852,SV-20785,Medium,US,371.976013,3,0.2,116.242996,22.030001,Standard Class,west,united states,santa ana,No
49668,CA-2013-160717,ME-17320,Medium,US,477.600006,3,0.2,161.190002,44.119999,Standard Class,west,united states,santa barbara,No


In [61]:
print('our data info: ')
print(df.shape)
print("\n")
print("\n")
print('our shipping mode: ')
print(df["ship_mode"].value_counts())

our data info: 
(49670, 14)




our shipping mode: 
ship_mode
Standard Class    29846
Second Class       9972
First Class        7232
Same Day           2620
Name: count, dtype: int64


In [62]:
outliers_sales = df[df['sales'] < 0]
outliers_discount = df[(df['discount'] < 0) | (df['discount'] > 1)]
outliers_shipping = df[df['shipping_cost'] < 0]

print("outliers sales:")
print(outliers_sales.shape[0])
print("outliers discount:")
print(outliers_discount.shape[0])
print("outliers shipping:")
print(outliers_shipping.shape[0])


high_sales = df[df['sales'] > df['sales'].quantile(0.99)]
print("Top 1% high sales orders:")
print(high_sales.head(10))


outliers sales:
0
outliers discount:
0
outliers shipping:
0
Top 1% high sales orders:
           order_id customer_id order_priority market        sales  quantity  \
156  CA-2014-143112    TS-21370         Medium     US  5199.959961         4   
158  CA-2011-145541    TB-21400         Medium     US  6999.959961         4   
219  CA-2013-128818    CJ-12010         Medium     US  3999.949951         5   
376  CA-2012-114811    KD-16495         Medium     US  4643.799805         4   
402  CA-2011-164973    NM-18445           High     US  3991.979980         2   
403  CA-2012-169796    Dp-13240           High     US  2321.899902         2   
450  CA-2014-165323    SR-20740           High     US  3404.500000         5   
459  CA-2014-168858    JD-16150           High     US  2504.739990         7   
554  CA-2014-127180    TA-21385           High     US  2399.600098         8   
565  CA-2011-160766    DM-13015           High     US  2799.959961         4   

     discount       profit  shipp

In [63]:
y = df['ship_mode']
X = df.drop(columns=['ship_mode', 'order_id', 'customer_id', 'city', 'profit', 'shipping_cost'])


num_cols = ['sales', 'quantity', 'discount']
cat_cols = [col for col in X.columns if col not in num_cols and df[col].nunique() < 15]


encoder = OneHotEncoder(drop="first", sparse_output=False)
X_encoded = pd.DataFrame(encoder.fit_transform(X[cat_cols]), columns=encoder.get_feature_names_out(cat_cols))


scaler = StandardScaler()
num_data_scaled = pd.DataFrame(scaler.fit_transform(X[num_cols]), columns=num_cols)

X_final = pd.concat([num_data_scaled.reset_index(drop=True), X_encoded.reset_index(drop=True)], axis=1)

X_train_full, X_test, y_train_full, y_test = train_test_split(X_final, y, test_size=0.2, random_state=42)


le = LabelEncoder()
y_train_encoded_full = le.fit_transform(y_train_full)
y_test_encoded = le.transform(y_test)


smote = SMOTE(random_state=42)
X_train, y_train_encoded = smote.fit_resample(X_train_full, y_train_encoded_full)


print(X_train.shape)
print( y_train_encoded.shape)


(95452, 25)
(95452,)


In [64]:
model = RandomForestClassifier(random_state=42)
model.fit(X_train, y_train_encoded)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [65]:
y_pred = model.predict(X_test)

In [66]:
print("accurate model:")
print(accuracy_score(y_test_encoded, y_pred))
print("\n")
print("all report:")
print(classification_report(y_test_encoded, y_pred))

accurate model:
0.4488624924501711


all report:
              precision    recall  f1-score   support

           0       0.23      0.28      0.25      1491
           1       0.09      0.17      0.12       512
           2       0.22      0.26      0.24      1948
           3       0.71      0.58      0.64      5983

    accuracy                           0.45      9934
   macro avg       0.31      0.32      0.31      9934
weighted avg       0.51      0.45      0.47      9934



In [67]:
xgb_model = XGBClassifier(random_state=42, eval_metric='mlogloss')
xgb_model.fit(X_train, y_train_encoded)


y_pred_xgb_encoded = xgb_model.predict(X_test)


y_pred_xgb = le.inverse_transform(y_pred_xgb_encoded)
y_test_labels = le.inverse_transform(y_test_encoded)


accuracy_xgb = accuracy_score(y_test_labels, y_pred_xgb)


print("accurate model:")
print(accuracy_xgb * 100)
print("\n")
print("all report:")
print(classification_report(y_test_labels, y_pred_xgb))


accurate model:
54.95268773907791


all report:
                precision    recall  f1-score   support

   First Class       0.28      0.32      0.30      1491
      Same Day       0.11      0.19      0.14       512
  Second Class       0.27      0.16      0.20      1948
Standard Class       0.74      0.76      0.75      5983

      accuracy                           0.55      9934
     macro avg       0.35      0.36      0.35      9934
  weighted avg       0.55      0.55      0.54      9934



In [68]:
df['sales_per_quantity'] = df['sales'] / df['quantity'].replace(0, 1e-6)
df['profit_margin'] = df['profit'] / df['sales'].replace(0, 1e-6)
df['shipping_cost_ratio'] = df['shipping_cost'] / df['sales'].replace(0, 1e-6)
df['profit_per_quantity'] = df['profit'] / df['quantity'].replace(0, 1e-6)

df['discount_level'] = pd.cut(df['discount'], bins=[0,0.05,0.15,0.3,1.0],
                              labels=['very_low','low','medium','high'])
df['sales_category'] = pd.qcut(df['sales'],4,labels=['Q1','Q2','Q3','Q4'],duplicates='drop')
df['profit_category'] = pd.qcut(df['profit'],4,labels=['low','medium','high','very_high'],duplicates='drop')

priority_map = {'High':3,'Medium':2,'Low':1,'Critical':4,'Not Specified':0}
df['order_priority_num'] = df['order_priority'].map(priority_map).fillna(0)
df['is_returned_num'] = df['isreturned'].map({'Yes':1,'No':0}).fillna(0)

num_cols = ['quantity','discount','sales','profit','shipping_cost','sales_per_quantity','profit_margin','shipping_cost_ratio','profit_per_quantity','order_priority_num','is_returned_num']
cat_cols = ['region','market','order_priority','discount_level','sales_category','profit_category']

X = df[num_cols + cat_cols]
y = df['ship_mode']

X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

enc = OneHotEncoder(drop="first",sparse_output=False,handle_unknown='ignore')
X_train_enc = pd.DataFrame(enc.fit_transform(X_train[cat_cols]),columns=enc.get_feature_names_out(cat_cols),index=X_train.index)
X_test_enc = pd.DataFrame(enc.transform(X_test[cat_cols]),columns=enc.get_feature_names_out(cat_cols),index=X_test.index)

sc = StandardScaler()
X_train_num = pd.DataFrame(sc.fit_transform(X_train[num_cols]),columns=num_cols,index=X_train.index)
X_test_num = pd.DataFrame(sc.transform(X_test[num_cols]),columns=num_cols,index=X_test.index)

X_train_final = pd.concat([X_train_num, X_train_enc],axis=1).astype(np.float32)
X_test_final = pd.concat([X_test_num, X_test_enc],axis=1).astype(np.float32)

le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

sm = SMOTE(random_state=42,k_neighbors=5,sampling_strategy='not majority')
X_train_bal, y_train_bal = sm.fit_resample(X_train_final,y_train_enc)

cw = compute_class_weight(class_weight='balanced',classes=np.unique(y_train_bal),y=y_train_bal)
w_dict = {i:v for i,v in enumerate(cw)}
s_weights = np.array([w_dict[i] for i in y_train_bal],dtype=np.float32)

model = XGBClassifier(random_state=42,n_estimators=400,learning_rate=0.02,max_depth=7,
                      min_child_weight=2,subsample=0.85,colsample_bytree=0.85,gamma=0.2,
                      reg_alpha=0.5,reg_lambda=2.0,eval_metric='mlogloss',
                      tree_method='hist',objective='multi:softmax',num_class=len(np.unique(y_train_enc)))

model.fit(X_train_bal,y_train_bal,sample_weight=s_weights,verbose=False)

preds_enc = model.predict(np.asarray(X_test_final,dtype=np.float32))
preds = le.inverse_transform(preds_enc)

print("XGBoost Accuracy:")
print("")
print(f"{accuracy_score(y_test,preds)*100:.2f}%")
print(classification_report(y_test,preds))


XGBoost Accuracy:

63.28%
                precision    recall  f1-score   support

   First Class       0.46      0.48      0.47      1446
      Same Day       0.14      0.08      0.10       524
  Second Class       0.32      0.28      0.30      1995
Standard Class       0.78      0.84      0.81      5969

      accuracy                           0.63      9934
     macro avg       0.43      0.42      0.42      9934
  weighted avg       0.61      0.63      0.62      9934



In [69]:
y_pred = preds  
y_test_str = y_test.astype(str)
y_pred_str = y_pred.astype(str)


order_ids = []
actual_modes = []
predicted_modes = []


for i in range(len(y_test_str)):
    order_id = df.loc[X_test_final.index[i], 'order_id'] if 'order_id' in df.columns else X_test_final.index[i]
    actual_mode = y_test_str.iloc[i]
    predicted_mode = y_pred_str[i]
    
    order_ids.append(order_id)
    actual_modes.append(actual_mode)
    predicted_modes.append(predicted_mode)


predictions = pd.DataFrame({
    'Order_ID': order_ids,
    'Actual_Ship_Mode': actual_modes,
    'Predicted_Ship_Mode': predicted_modes
})

predictions.to_csv('predictions_Ship_Mode.csv', index=False)


report = classification_report(y_test_str, y_pred_str, output_dict=True)
report_df = pd.DataFrame(report).transpose()
report_df.index.name = 'Class'
report_df.to_csv('model_classification_report_Ship_Mode.csv', index=True)


params = model.get_params()


model_df = pd.DataFrame({
    'Model': ['XGBoost'],
    'Accuracy (%)': [round(accuracy_score(y_test_str, y_pred_str) * 100, 2)],
    'n_estimators': [params['n_estimators']],
    'learning_rate': [params['learning_rate']],
    'max_depth': [params['max_depth']],
    'min_child_weight': [params['min_child_weight']],
    'subsample': [params['subsample']],
    'colsample_bytree': [params['colsample_bytree']],
    'gamma': [params['gamma']],
    'reg_alpha': [params['reg_alpha']],
    'reg_lambda': [params['reg_lambda']],
    'tree_method': [params['tree_method']],
    'objective': [params['objective']]
})

model_df.to_csv('model_accuracy_with_params.csv', index=False)